# Linked Lists — Technical Reference

*Reference — flat lookup, no prose. Scan or ctrl+F when you need a specific manipulation mid-problem.*

## Quick Index

| Technique | When to use | Problems |
| :--- | :--- | :--- |
| Dummy Node | Simplify insert/delete at head; avoid null checks | 21, 82, 203 |
| In-place Reversal | Reverse full list or sublist without extra space | 206, 92, 25 |
| Fast & Slow Pointers | Cycle detection, find middle, kth from end | 141, 142, 876 |
| Merge Two Sorted Lists | Combine two sorted lists in O(n + m) | 21, 23 |
| Find Intersection | Two lists meet at a node — equalize lengths | 160 |

## When to Use

| Signal | Technique to reach for |
| :--- | :--- |
| Operation may change the head node | Dummy node |
| "reverse", "swap in pairs", "rotate" | In-place reversal |
| "does it have a cycle", "find the middle", "nth from end" | Fast & slow pointers |
| "merge", "sort a linked list" | Merge two sorted lists (+ fast/slow to split) |
| Two lists that may share a suffix | Find intersection |
| O(1) space demanded on a list problem | Pointer rewiring, never an array copy |

---
## Dummy Node

Create a throwaway node before the head. Eliminates special cases for operations on the head node — the real head is always `dummy.next`.

In [ ]:
# Template — dummy node
dummy = ListNode(0)
dummy.next = head
cur = dummy
# ... manipulate list using cur ...
return dummy.next               # real head may have changed


# LC 82 — Remove Duplicates from Sorted List II
# Remove all nodes whose value appears more than once
def deleteDuplicates(head):
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        cur = prev.next
        if cur.next and cur.val == cur.next.val:
            while cur.next and cur.val == cur.next.val:
                cur = cur.next          # skip all nodes with this value
            prev.next = cur.next        # bypass the duplicate group
        else:
            prev = prev.next
    return dummy.next

Problems: LC 21, LC 82, LC 203

---
## In-place Reversal

Reverse a linked list using three pointers: `prev`, `cur`, `next_node`. No extra space. For sublist reversal, locate the sublist boundaries first using a dummy node.

In [ ]:
# LC 206 — Reverse Linked List
def reverseList(head):
    prev, cur = None, head
    while cur:
        next_node = cur.next    # save next before overwriting
        cur.next  = prev        # reverse pointer
        prev      = cur         # advance prev
        cur       = next_node   # advance cur
    return prev                 # prev is new head


# LC 92 — Reverse Linked List II (reverse sublist from left to right)
def reverseBetween(head, left, right):
    dummy = ListNode(0, head)
    prev  = dummy
    for _ in range(left - 1):      # move prev to node just before sublist
        prev = prev.next

    cur = prev.next
    for _ in range(right - left):  # reverse (right - left) times
        next_node    = cur.next
        cur.next     = next_node.next
        next_node.next = prev.next
        prev.next    = next_node
    return dummy.next

Problems: LC 206, LC 92, LC 25

---
## Fast & Slow Pointers

`fast` moves two steps per iteration, `slow` moves one. If a cycle exists they meet inside it. When `fast` reaches null, `slow` is at the middle.

In [ ]:
# Cycle detection
def hasCycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast: return True
    return False

# Find middle (slow is at middle when fast hits end)
def findMiddle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

# Find kth node from end (advance fast k steps first)
def kthFromEnd(head, k):
    slow = fast = head
    for _ in range(k):
        fast = fast.next        # fast is k steps ahead
    while fast:
        slow = slow.next
        fast = fast.next
    return slow                 # when fast hits null, slow is k from end

Problems: LC 141, LC 142, LC 876, LC 19

---
## Merge Two Sorted Lists

Use a dummy node as the anchor. At each step, append the smaller of the two current nodes. After one list is exhausted, attach the remainder of the other.

In [ ]:
# LC 21 — Merge Two Sorted Lists
def mergeTwoLists(l1, l2):
    dummy = cur = ListNode(0)
    while l1 and l2:
        if l1.val <= l2.val:
            cur.next = l1
            l1 = l1.next
        else:
            cur.next = l2
            l2 = l2.next
        cur = cur.next
    cur.next = l1 or l2         # attach remaining nodes
    return dummy.next

Problems: LC 21, LC 23 (K-way merge — see Heap cheat sheet)

---
## Find Intersection

Two pointers walk both lists. When one reaches the end, redirect it to the head of the other list. After at most `len(A) + len(B)` steps both pointers are equidistant from the intersection — they meet at it, or both hit null together.

In [ ]:
# LC 160 — Intersection of Two Linked Lists
def getIntersectionNode(headA, headB):
    a, b = headA, headB
    while a != b:
        a = a.next if a else headB  # redirect to other list's head at end
        b = b.next if b else headA
    return a                        # intersection node, or None if no intersection

Problems: LC 160

---
## Common mistakes

| Mistake | What you observe | Fix |
| :--- | :--- | :--- |
| Returning `head` instead of `dummy.next` | Wrong answer whenever the head itself is removed | The real head may have changed — return `dummy.next` |
| Losing `cur.next` before reassigning it | Truncated list, or an infinite loop | Save `next_node = cur.next` *before* overwriting `cur.next` |
| Checking `fast.next.next` without checking `fast.next` | `AttributeError` on even-length lists | Guard with `while fast and fast.next` |
| Returning `slow` for "second middle" when the problem wants the first | Off by one on even-length lists | Even length gives two middles — decide which and adjust the loop |
| Creating a cycle while rewiring | Test hangs, or times out | After rewiring, make sure the tail's `next` is set to `None` |
| Comparing node values instead of node identity | Wrong answer when duplicate values exist | Use `is` / `==` on nodes, not on `.val` |
| Forgetting `prev.next` still points at the removed node | Node not actually deleted | Advance `prev` only when you did not delete |
| Reversing a sublist without anchoring the node before it | Sublist detached from the list | Locate `prev` of the sublist first, using a dummy node |
| Assuming both lists are the same length in intersection | Returns `None` on a real intersection | The redirect trick equalises lengths automatically — do not add manual offsets |
| Converting the list to an array "for convenience" | Passes, but forfeits the O(1) space the problem wanted | Rewire pointers in place |